## RL Training Notebook

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np

from src.data.loader import RossmannDataLoader
from src.data.preprocessor import RossmannPreprocessor
from src.data.feature_engineer import FeatureEngineer
from src.rl.curriculum import CurriculumScheduler
from src.rl.inventory_env import RetailInventoryEnv
from src.rl.reward import ShapedReward
from src.rl.agent import RLInventoryAgent

In [ ]:
loader = RossmannDataLoader('../data/raw')
train_df, _ = loader.load()
pre = RossmannPreprocessor()
train_df = pre.fit_transform(train_df)
fe = FeatureEngineer()
train_df = fe.transform(train_df)
print(train_df.shape)

In [ ]:
scheduler = CurriculumScheduler(train_df, n_levels=4)
reward_fn = ShapedReward(holding_cost=0.5, stockout_cost=5.0, order_cost=10.0, gamma=0.99, target_stock=120.0)

store_id = scheduler.current_store_ids()[0]
store_sales = train_df[train_df['Store'] == store_id].sort_values('Date')['Sales'].values.astype(float)
env = RetailInventoryEnv(config_path='../configs/rl_config.yaml', demand_series=store_sales, reward_fn=reward_fn)
agent = RLInventoryAgent(config_path='../configs/rl_config.yaml', env=env)
agent.build()
print('Prepared agent for store', store_id)

In [ ]:
# For quick experimentation, reduce total_timesteps in configs/rl_config.yaml first.
# agent.train()
print('Run agent.train() after lowering timesteps for notebook runtime.')

In [ ]:
obs, _ = env.reset()
reward_trace = []
for _ in range(30):
    action = np.array([0.0], dtype=float)
    obs, reward, terminated, truncated, info = env.step(action)
    reward_trace.append(float(reward))
    if terminated or truncated:
        break
print('Mean baseline reward over 30 steps:', float(np.mean(reward_trace)))